# 2. Ingeniería de Características (Feature Engineering)

En la ingeniería de características tradicional, solíamos mapear manualmente las columnas categóricas en DataFrames de pandas (por ejemplo, convirtiendo `Sex` de 'male'/'female' a 0/1, o aplicando One-Hot encoding manualmente en archivos intermedios como `data/titanic_feature.csv`).

Sin embargo, este enfoque manual tiene desventajas graves en producción:
- Propensión a errores por inconsistencias de categorías en tiempo real.
- Dificultad para mantener scripts paralelos de preprocesamiento.
- Riesgo de fuga de información (data leakage) al escalar los datos antes del split.

### Enfoque Moderno: Pipelines Automatizados

En lugar de crear archivos de datos preprocesados manualmente, utilizaremos el **ColumnTransformer** y **Pipeline** de `scikit-learn`. Este enfoque tiene grandes ventajas:
1. **Automatización Completa**: Cuando el frontend envía los datos crudos en formato JSON (ej. `Sex='female'`, `Embarked='C'`), el preprocesador empaquetado realiza la codificación One-Hot y el escalamiento estándar automáticamente en memoria.
2. **Previene Fugas**: El escalamiento estándar (`StandardScaler`) calcula la media y la desviación estándar **solo en el conjunto de entrenamiento**, y las aplica al de prueba, garantizando una correcta evaluación.
3. **Código Limpio**: Evita tener que escribir funciones de mapeo personalizadas en la API de Flask, haciendo que `app.py` sea sumamente conciso.

In [ ]:
# Demostración del Preprocesamiento del Pipeline
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Cargar los datos limpios
df = pd.read_csv('data/titanic_clean.csv')
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]

# Definir columnas
numerical_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_cols = ['Sex', 'Embarked']

# Crear el transformador de columnas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Ajustar y transformar una muestra para ver los resultados mapeados
X_transformed = preprocessor.fit_transform(X)
print("Datos originales (primer registro):")
print(X.iloc[0])
print("\nDatos transformados por el preprocesador (primer registro):\n", X_transformed[0])

# 2.3 Feature Engineering

**Glosario**

Feature engineering es el proceso de crear nuevas variables (features) a partir de los datos existentes y transformar valores de variables actuales para mejorar el rendimiento de un modelo de machine learning. Esto incluye técnicas como normalización, escalado, codificación de variables categóricas y creación de nuevas variables derivadas de las existentes.

## Importar paquetes

Estamos en un nuevo notebook, entonces tenemos que volver a importar los paquetes que usaremos:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import QuantileTransformer

## Carga de datos

En el notebook anterior guardamos nuestros datos limpios en el directorio `data/`. Carguemos estos datos a DataFrames.

In [ ]:
df = pd.read_csv('./data/titanic_clean.csv')
df.head()

## Feature Engineering

Al momento de entrenar un modelo de machine learning, es común que tengamos la necesidad de crear nuevas variables. Un ejemplo de esto podría ser en un modelo de predicción de ventas, donde podríamos crear una nueva variable que represente el día de la semana a partir de una columna de fechas, ya que las ventas pueden variar significativamente entre días laborables y fines de semana.

**Antes de feature engineering:**

| id_venta | fecha_venta | valor_venta |
|----------|-------------|-------------|
| 2050423  | 2024-07-01  | 5400        |
| 3423434  | 2024-07-06  | 3844        |
| 4994933  | 2024-03-05  | 9200        |

**Después de feature engineering:**

| id_venta | fecha_venta | dia_venta  | valor_venta |
|----------|-------------|------------|-------------|
| 2050423  | 2024-07-01  | lunes      | 5400        |
| 3423434  | 2024-07-08  | lunes      | 3844        |
| 4994933  | 2024-03-06  | miércoles  | 9200        |

En nuestro caso, no crearemos nuevas variables, pero sí realizaremos transformaciones a nuestros datos que harán que nuestros modelos sean más precisos. Estas transformaciones son muy comunes en modelos de machine learning.

## Codificación de variables categóricas

La codificación de variables categóricas es el proceso de convertir datos categóricos (como colores o tipos de productos) en números para que los modelos de machine learning puedan utilizarlos. Por ejemplo, convertir "rojo", "verde" y "azul" en 1, 2 y 3.

Debemos recordar que los modelos de machine learning son **funciones matemáticas**, lo que implica que los datos de entrada deben ser números. Nuestro conjunto de datos tiene algunas variables categóricas:

- `Sex`
- `Embarked`

### Scikit Learn LabelEncoder

Scikit Learn es la librería por excelencia para hacer aprendizaje automático en Python. Este paquete cuenta con una clase `LabelEncoder` que se utiliza justamente para convertir variables categóricas a numéricas.

¡Y es muy fácil usarla!

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Datos categóricos de ejemplo
animales = ['gato', 'perro', 'pez', 'gato', 'perro']

# Crear una instancia de LabelEncoder
le = LabelEncoder()

# Ajustar y transformar los datos
animales_codificados = le.fit_transform(animales)

# Mostrar los datos codificados
print(animales_codificados)

# Si deseas ver cómo se mapean las etiquetas originales
print(list(le.classes_))

### Codificación de Sex y Embarked

In [ ]:
label_sex = LabelEncoder()
label_embarked = LabelEncoder()

# Fit and transform on training data
df['Sex'] = label_sex.fit_transform(df['Sex'])
df['Embarked'] = label_embarked.fit_transform(df['Embarked'])

df.sample(5)

¡Ya no tenemos variables categóricas!

## Normalización

Adicional al proceso de codificación de variables categóricas, un paso muy útil y muy común en Feature Engineering es la normalización.

> **¿Qué es la normalización?**
> La normalización en Feature Engineering es un proceso que ajusta los valores de las características (features) de los datos para que estén dentro de un rango común, típicamente entre 0 y 1. Esto se hace para mejorar el rendimiento y la velocidad de los algoritmos de aprendizaje automático, asegurando que todas las características tengan la misma escala y no dominen unas sobre otras.

> **¿Por qué mejora el rendimiento?**
> - **Convergencia más rápida**: Los algoritmos de optimización como el descenso de gradiente convergen más rápido cuando las características están en la misma escala.
> - **Estabilidad numérica**: Ayuda a evitar problemas de estabilidad numérica durante los cálculos.
> - **Mejora de la precisión**: Permite que los algoritmos consideren todas las características por igual.
> - **Uniformidad en métricas de distancia**: En algoritmos como k-vecinos más cercanos o SVM, la normalización asegura que ninguna característica afecte desproporcionadamente las medidas de distancia.

> **¿Tiene que ver con la distribución normal?**
> En el contexto de Feature Engineering, "normalización" **no** se refiere a la distribución normal (gaussiana). Hay dos conceptos distintos:
> - **Normalización (Min-Max Scaling)**: Escala los datos para que caigan dentro de un rango específico como [0, 1].
> - **Estandarización (Z-score Scaling)**: Transforma los datos para que tengan media 0 y desviación estándar 1.

Tenemos dos variables numéricas cuyos rangos y distribución no son ideales:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['Age'].dropna(), kde=True, ax=axes[0])
axes[0].set_title('Distribución de Age')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frecuencia')

sns.histplot(df['Fare'].dropna(), kde=True, ax=axes[1])
axes[1].set_title('Distribución de Fare')
axes[1].set_xlabel('Fare')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

La distribución de `Age` se muestra distribuida aproximadamente de manera normal entre 0 y 80 años, y `Fare` altamente sesgada con muchos valores bajos y algunos valores atípicos elevados. Normalizar estas variables nos servirá porque ajusta los valores a un rango común, mejorando la convergencia de los algoritmos que implementaremos.

## Scikit-Learn QuantileTransformer

El `QuantileTransformer` es una herramienta de Scikit-Learn que transforma los datos para que sigan una distribución uniforme o normal. Esto se logra ajustando los valores de los datos según sus cuantiles, lo que puede reducir el impacto de los valores atípicos y mejorar el rendimiento de los algoritmos de aprendizaje automático.

Usar `QuantileTransformer` para normalizar `Age` y `Fare` es una buena idea porque transforma las características a una distribución uniforme o normal, lo cual es especialmente útil para manejar características con distribuciones muy sesgadas o con valores atípicos. Dado que `Fare` tiene una distribución altamente sesgada con una larga cola, `QuantileTransformer` puede redistribuir los datos de manera más equitativa.

In [ ]:
qun_tra_age  = QuantileTransformer(output_distribution='normal', n_quantiles=500)
qun_tra_fare = QuantileTransformer(output_distribution='normal', n_quantiles=500)

df['Age']  = qun_tra_age.fit_transform(df[['Age']])
df['Fare'] = qun_tra_fare.fit_transform(df[['Fare']])

Después de ejecutar este código, volvamos a generar las gráficas de las distribuciones:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['Age'].dropna(), kde=True, ax=axes[0])
axes[0].set_title('Distribución de Age (después de QuantileTransformer)')
axes[0].set_xlabel('Age (normalizada)')
axes[0].set_ylabel('Frecuencia')

sns.histplot(df['Fare'].dropna(), kde=True, ax=axes[1])
axes[1].set_title('Distribución de Fare (después de QuantileTransformer)')
axes[1].set_xlabel('Fare (normalizada)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

Ahora ambas variables siguen una distribución normal. Nota cómo `Age` ya no cuenta con valores de entre 0 y 80 años, sino entre -5 y 5 con una media de 0.

## Escalado

Nos queda un paso más en nuestro proceso de Feature Engineering.

El proceso de ajustar los valores de las características a un rango específico se llama **escalado**, y es necesario por varias razones:

- **Equilibrio de características:** En un dataset, diferentes características pueden tener diferentes escalas. Por ejemplo, `Age` puede variar de 0 a 80, mientras que `Fare` puede variar de 0 a 500. Sin escalado, los algoritmos pueden dar más peso a las características con valores más grandes.

- **Mejor rendimiento del modelo:** Los algoritmos de optimización como el descenso de gradiente convergen más rápido y de manera más estable cuando las características están en la misma escala.

- **Uniformidad en las métricas de distancia:** En algoritmos como KNN y SVM, el escalado asegura que ninguna característica domine la métrica de distancia debido a su magnitud.

## MinMaxScaler

El `MinMaxScaler` ajusta los valores de los datos para que estén dentro de un rango definido, típicamente entre 0 y 1. Este escalado lineal usa la fórmula:

$$X_{escalado} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

donde $X$ es el valor original, $X_{min}$ es el valor mínimo en la característica y $X_{max}$ es el valor máximo. Esto asegura que todas las características contribuyan equitativamente al proceso de modelado.

In [ ]:
mms_pclass   = MinMaxScaler()
mms_sex      = MinMaxScaler()
mms_age      = MinMaxScaler()
mms_sibsp    = MinMaxScaler()
mms_parch    = MinMaxScaler()
mms_fare     = MinMaxScaler()
mms_embarked = MinMaxScaler()

df['Pclass']   = mms_pclass.fit_transform(df[['Pclass']])
df['Sex']      = mms_sex.fit_transform(df[['Sex']])
df['Age']      = mms_age.fit_transform(df[['Age']])
df['SibSp']    = mms_sibsp.fit_transform(df[['SibSp']])
df['Parch']    = mms_parch.fit_transform(df[['Parch']])
df['Fare']     = mms_fare.fit_transform(df[['Fare']])
df['Embarked'] = mms_embarked.fit_transform(df[['Embarked']])

df.sample(5)

> 💡 **Prompt para ChatGPT:**
> 
> *"Explícame este código línea por línea:"*
> 
> ```python
> mms_pclass = MinMaxScaler()
> df['Pclass'] = mms_pclass.fit_transform(df[['Pclass']])
> ```
> 
> ChatGPT puede explicarte por qué se crea una instancia separada por columna y la diferencia entre `df[['Pclass']]` (DataFrame) y `df['Pclass']` (Series).

## Guardado del dataset procesado

Guardamos el DataFrame con todas las transformaciones aplicadas para usarlo en el siguiente notebook de Machine Learning.

In [ ]:
df.to_csv('./data/titanic_procesado.csv', index=False)
print("Dataset guardado en: ./data/titanic_procesado.csv")
print(f"Shape: {df.shape}")
df.head()

## StandardScaler (Estandarización Z-score)

`StandardScaler` transforma los datos para que tengan **media = 0** y **desviación estándar = 1**. La fórmula es:

$$z = \frac{x - \mu}{\sigma}$$

Donde:
- $x$ = valor original
- $\mu$ = media de la columna
- $\sigma$ = desviación estándar de la columna

Es ideal cuando los datos siguen aproximadamente una distribución normal y cuando el algoritmo asume que las variables están centradas en cero (ej. Regresión Logística, SVM, PCA).

In [ ]:
scaler = StandardScaler()

df_standard = df.copy()
df_standard[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

print("Antes del escalado:")
print(df[['Age', 'Fare']].describe().round(2))
print("\nDespués de StandardScaler:")
print(df_standard[['Age', 'Fare']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df_standard['Age'].dropna(), kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age — StandardScaler')
axes[0].set_xlabel('Age (z-score)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(0, color='red', linestyle='--', label='Media = 0')
axes[0].legend()

sns.histplot(df_standard['Fare'].dropna(), kde=True, ax=axes[1], color='mediumpurple')
axes[1].set_title('Fare — StandardScaler')
axes[1].set_xlabel('Fare (z-score)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(0, color='red', linestyle='--', label='Media = 0')
axes[1].legend()

plt.tight_layout()
plt.show()

## MinMaxScaler (Normalización Min-Max)

`MinMaxScaler` escala los valores al rango **[0, 1]**. La fórmula es:

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Donde:
- $x_{min}$ = valor mínimo de la columna
- $x_{max}$ = valor máximo de la columna

Es útil cuando se necesita un rango acotado y conocido. Sin embargo, es **sensible a outliers**: un valor extremo en `Fare` empujará todos los demás valores hacia 0.

In [ ]:
minmax = MinMaxScaler()

df_minmax = df.copy()
df_minmax[['Age', 'Fare']] = minmax.fit_transform(df[['Age', 'Fare']])

print("Después de MinMaxScaler:")
print(df_minmax[['Age', 'Fare']].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df_minmax['Age'].dropna(), kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age — MinMaxScaler')
axes[0].set_xlabel('Age (0–1)')
axes[0].set_ylabel('Frecuencia')

sns.histplot(df_minmax['Fare'].dropna(), kde=True, ax=axes[1], color='mediumpurple')
axes[1].set_title('Fare — MinMaxScaler')
axes[1].set_xlabel('Fare (0–1)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## Comparación: Original vs StandardScaler vs MinMaxScaler

| | Age | Fare |
|---|---|---|
| **Original** | 0.42 – 80 | 0 – 512 |
| **StandardScaler** | media≈0, std≈1 | media≈0, std≈1 |
| **MinMaxScaler** | 0 – 1 | 0 – 1 |

**¿Cuál usar?**
- `StandardScaler` → Regresión Logística, SVM, PCA, redes neuronales
- `MinMaxScaler` → Redes neuronales con activación sigmoid/tanh, cuando el rango [0,1] es necesario
- `QuantileTransformer` → Variables muy sesgadas como `Fare` (convierte la distribución a uniforme o normal)

## QuantileTransformer

`QuantileTransformer` es especialmente útil para variables con distribuciones muy sesgadas, como `Fare`. A diferencia de `StandardScaler` y `MinMaxScaler`, **no es una transformación lineal**: reordena los valores según su posición en la distribución (cuantiles).

Tiene dos modos de salida (`output_distribution`):

- **`'uniform'`**: transforma los datos a una distribución uniforme en [0, 1]
- **`'normal'`**: transforma los datos para que sigan aproximadamente una distribución normal (gaussiana)

Es robusto ante outliers porque se basa en rangos, no en la magnitud de los valores.

In [ ]:
qt_uniform = QuantileTransformer(output_distribution='uniform', n_quantiles=100, random_state=42)
qt_normal  = QuantileTransformer(output_distribution='normal',  n_quantiles=100, random_state=42)

df_qt_uniform = df.copy()
df_qt_normal  = df.copy()

df_qt_uniform[['Age', 'Fare']] = qt_uniform.fit_transform(df[['Age', 'Fare']])
df_qt_normal[['Age', 'Fare']]  = qt_normal.fit_transform(df[['Age', 'Fare']])

print("QuantileTransformer (uniform) — Fare:")
print(df_qt_uniform['Fare'].describe().round(4))
print("\nQuantileTransformer (normal) — Fare:")
print(df_qt_normal['Fare'].describe().round(4))

### Comparación visual: Fare original vs las 4 transformaciones

Observa cómo `Fare` pasa de una distribución extremadamente sesgada a distribuciones más manejables:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, QuantileTransformer

df = pd.read_csv('./data/titanic_clean.csv')
df['Sex']      = LabelEncoder().fit_transform(df['Sex'])
df['Embarked'] = LabelEncoder().fit_transform(df['Embarked'])

df_std     = df.copy(); df_std[['Age','Fare']]     = StandardScaler().fit_transform(df[['Age','Fare']])
df_mm      = df.copy(); df_mm[['Age','Fare']]      = MinMaxScaler().fit_transform(df[['Age','Fare']])
df_qt_uni  = df.copy(); df_qt_uni[['Age','Fare']]  = QuantileTransformer(output_distribution='uniform', n_quantiles=100, random_state=42).fit_transform(df[['Age','Fare']])
df_qt_norm = df.copy(); df_qt_norm[['Age','Fare']] = QuantileTransformer(output_distribution='normal',  n_quantiles=100, random_state=42).fit_transform(df[['Age','Fare']])

fig, axes = plt.subplots(1, 5, figsize=(24, 5))

datasets = [
    (df,         'Fare — Original',           '#888888'),
    (df_std,     'Fare — StandardScaler',      'steelblue'),
    (df_mm,      'Fare — MinMaxScaler',        'coral'),
    (df_qt_uni,  'Fare — Quantile (uniform)',  'mediumseagreen'),
    (df_qt_norm, 'Fare — Quantile (normal)',   'mediumpurple'),
]

for ax, (data, title, color) in zip(axes, datasets):
    sns.histplot(data['Fare'].dropna(), kde=True, ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel('Fare')
    ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## Guardado del dataset transformado

Ahora que exploramos las distintas transformaciones, debemos elegir **una sola** para guardar el dataset final que usaremos en el entrenamiento.

### ¿Qué scaler elegimos?

Para este proyecto usaremos **`StandardScaler`** por las siguientes razones:

| Scaler | Adecuado para Titanic | Razón |
|--------|----------------------|-------|
| `StandardScaler` | ✅ **Sí** | Funciona bien con Regresión Logística y Random Forest; robusto en la mayoría de algoritmos |
| `MinMaxScaler` | ⚠️ Parcialmente | Sensible a los outliers de `Fare` (máx 512); comprime la mayoría de valores cerca de 0 |
| `QuantileTransformer` | ✅ Sí | Excelente para `Fare`, pero introduce no-linealidad que puede complicar la interpretación |

`StandardScaler` es el estándar de la industria para estos modelos y será el que usaremos.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Cargar datos limpios desde cero para evitar transformaciones acumuladas
df_final = pd.read_csv('./data/titanic_clean.csv')

# 1. Codificación de variables categóricas
label_sex      = LabelEncoder()
label_embarked = LabelEncoder()
df_final['Sex']      = label_sex.fit_transform(df_final['Sex'])
df_final['Embarked'] = label_embarked.fit_transform(df_final['Embarked'])

# 2. Escalado de variables numéricas continuas
scaler = StandardScaler()
df_final[['Age', 'Fare']] = scaler.fit_transform(df_final[['Age', 'Fare']])

# 3. Guardar
df_final.to_csv('./data/titanic_feature.csv', index=False)

print("Dataset guardado en: ./data/titanic_feature.csv")
print(f"Shape: {df_final.shape}")
print("\nPrimeras 5 filas:")
df_final.head()

### Verificación del dataset guardado

Confirmemos que el archivo se guardó correctamente y que no quedan variables categóricas:

In [ ]:
df_verificacion = pd.read_csv('./data/titanic_feature.csv')

print("Tipos de datos:")
print(df_verificacion.dtypes)
print(f"\nValores nulos:\n{df_verificacion.isnull().sum()}")
print(f"\nEstadísticos de Age y Fare (deben tener media ≈ 0):")
print(df_verificacion[['Age', 'Fare']].describe().round(4))